In [1]:

!pip install -q transformers gradio soundfile sentencepiece huggingface_hub

In [2]:

import torch
import numpy as np
import soundfile as sf
import gradio as gr
from huggingface_hub import hf_hub_download
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan

In [3]:
from huggingface_hub import hf_hub_download

zip_path = hf_hub_download(
    repo_id="Matthijs/cmu-arctic-xvectors",
    filename="spkrec-xvect.zip",
    repo_type="dataset",
)

spkrec-xvect.zip:   0%|          | 0.00/17.9M [00:00<?, ?B/s]

In [4]:
import zipfile
import os

extract_path = "/kaggle/working/xvectors"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(os.listdir(extract_path))

['spkrec-xvect']


In [5]:
import soundfile as sf
import torch

SAMPLE_RATE = 16000
OUTPUT_PATH = "/kaggle/working/tts_output.wav"

def synthesize(text, speaker_name):
    if not text or not text.strip():
        return None, "Please enter some text."

    # Tokenize
    inputs = processor(text=text, return_tensors="pt")
    input_ids = inputs["input_ids"]

    # Trim long inputs (SpeechT5 limit)
    if input_ids.shape[-1] > 600:
        input_ids = input_ids[:, :600]

    # Get speaker embedding
    speaker_embeddings = get_speaker_embedding(speaker_name).to(device)

    # Generate speech
    with torch.no_grad():
        speech = model.generate_speech(
            input_ids.to(device),
            speaker_embeddings,
            vocoder=vocoder
        )

    # Convert to numpy safely
    audio_np = speech.cpu().numpy()

    # 🔥 Normalize audio (IMPORTANT improvement)
    audio_np = audio_np / max(abs(audio_np))

    # Save file
    sf.write(OUTPUT_PATH, audio_np, SAMPLE_RATE)

    return OUTPUT_PATH, "Done!"

In [6]:
import torch
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

# ── Speaker embeddings mapping ───────────────────────────────────────────────
SPEAKERS = {
    "Female (slt)": "/kaggle/working/xvectors/spkrec-xvect/cmu_us_slt_arctic-wav-arctic_a0280.npy",
    "Male (bdl)"  : "/kaggle/working/xvectors/spkrec-xvect/cmu_us_bdl_arctic-wav-arctic_a0126.npy",
    "Male (awb)"  : "/kaggle/working/xvectors/spkrec-xvect/cmu_us_awb_arctic-wav-arctic_a0181.npy",
    "Male (rms)"  : "/kaggle/working/xvectors/spkrec-xvect/cmu_us_rms_arctic-wav-arctic_a0093.npy",
}

# ── Function to load speaker embedding ─────────────────────────────────────
def get_speaker_embedding(speaker_name):
    file_path = SPEAKERS[speaker_name]
    xvec = np.load(file_path)
    return torch.tensor(xvec).unsqueeze(0).to(device)

In [7]:
speaker_dropdown = gr.Dropdown(
    choices=list(SPEAKERS.keys()),
    value=list(SPEAKERS.keys())[0],  # safest: pick first speaker
    label="Speaker voice"
)

In [8]:
import torch
import numpy as np
import soundfile as sf
import gradio as gr
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech
from transformers import SpeechT5HifiGan

# ── Device ─────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"

# ── Load processor + TTS model + vocoder ────────────────
processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model     = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts").to(device)
vocoder   = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").to(device)

# ── Speaker embeddings ──────────────────────────────────
SPEAKERS = {
    "Female (slt)": "/kaggle/working/xvectors/spkrec-xvect/cmu_us_slt_arctic-wav-arctic_a0280.npy",
    "Male (bdl)"  : "/kaggle/working/xvectors/spkrec-xvect/cmu_us_bdl_arctic-wav-arctic_a0126.npy",
    "Male (awb)"  : "/kaggle/working/xvectors/spkrec-xvect/cmu_us_awb_arctic-wav-arctic_a0181.npy",
    "Male (rms)"  : "/kaggle/working/xvectors/spkrec-xvect/cmu_us_rms_arctic-wav-arctic_a0093.npy",
}

def get_speaker_embedding(speaker_name):
    file_path = SPEAKERS[speaker_name]
    xvec = np.load(file_path)
    return torch.tensor(xvec).unsqueeze(0).to(device)

# ── TTS function ────────────────────────────────────────
SAMPLE_RATE = 16000
OUTPUT_PATH = "/kaggle/working/tts_output.wav"

def synthesize(text, speaker_name):
    if not text or not text.strip():
        return None, "Please enter some text."

    inputs = processor(text=text, return_tensors="pt")
    input_ids = inputs["input_ids"]

    if input_ids.shape[-1] > 600:
        input_ids = input_ids[:, :600]

    speaker_embeddings = get_speaker_embedding(speaker_name)

    with torch.no_grad():
        speech = model.generate_speech(
            input_ids.to(device),
            speaker_embeddings,
            vocoder=vocoder
        )

    audio_np = speech.cpu().numpy()
    sf.write(OUTPUT_PATH, audio_np, SAMPLE_RATE)
    return OUTPUT_PATH, "Done!"

preprocessor_config.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/232 [00:00<?, ?B/s]

spm_char.model:   0%|          | 0.00/238k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/585M [00:00<?, ?B/s]

SpeechT5ForTextToSpeech LOAD REPORT from: microsoft/speecht5_tts
Key                                         | Status     |  | 
--------------------------------------------+------------+--+-
speecht5.decoder.prenet.encode_positions.pe | UNEXPECTED |  | 
speecht5.encoder.prenet.encode_positions.pe | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/50.7M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/158 [00:00<?, ?it/s]

In [9]:

with gr.Blocks(title="SpeechT5 Text-to-Speech") as demo:
    gr.Markdown("## 🔊 Text-to-Speech with SpeechT5")
    gr.Markdown("Type your text, choose a speaker voice, and hit **Synthesize**.")

    with gr.Row():
        text_input = gr.Textbox(
            label="Input text",
            placeholder="Enter text here (max ~600 tokens)...",
            lines=4
        )

    with gr.Row():
        speaker_dropdown = gr.Dropdown(
            choices=list(SPEAKERS.keys()),
            value="Speaker 1 (female)",
            label="Speaker voice"
        )
        run_btn = gr.Button("🔊 Synthesize", variant="primary")

    with gr.Row():
        audio_output  = gr.Audio(label="Generated speech", type="filepath")
        status_output = gr.Textbox(label="Status", interactive=False)

    run_btn.click(
        fn=synthesize,
        inputs=[text_input, speaker_dropdown],
        outputs=[audio_output, status_output]
    )

demo.launch(share=True, debug=False)

/usr/local/lib/python3.12/dist-packages/gradio/components/dropdown.py:230: UserWarning: The value passed into gr.Dropdown() is not in the list of choices. Please update the list of choices to include: Speaker 1 (female) or set allow_custom_value=True.
  warnings.warn(


model.safetensors:   0%|          | 0.00/50.6M [00:00<?, ?B/s]

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://b8a26e3709ff8bda97.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
